<a href="https://colab.research.google.com/github/Houcineaberhache/nanoGPT-Scaling-Laws/blob/feature%2Ftraining/train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Training Notebook — Training Engineer

**Owner:** Houcine (Student 3 — Training Engineer)
**Scope:** Colab + GPU setup → launch the 6 experiments (E1–E6) → save checkpoints to Drive → record losses and training time.

| Run | Model | Data |
|---|---|---|
| E1 / E2 / E3 | Small | 25 % / 50 % / 100 % |
| E4 / E5 / E6 | Medium | 25 % / 50 % / 100 % |

> 📊 Test-set evaluation, plots, text analysis and scaling laws belong to the **Evaluation Engineer** → see `notebooks/evaluation_colab.ipynb`.

**Before running:** `Runtime ▸ Change runtime type ▸ T4 GPU`.

## 1. Environment setup

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name)
    print("GPU memory:", round(props.total_memory / 1024**3, 2), "GB")
else:
    raise RuntimeError("No GPU detected. Runtime > Change runtime type > T4 GPU")

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [2]:
%cd /content

import os
if not os.path.exists("nanoGPT-Scaling-Laws"):
    !git clone https://github.com/Houcineaberhache/nanoGPT-Scaling-Laws.git

%cd /content/nanoGPT-Scaling-Laws
!git checkout main
!git pull origin main

/content
Cloning into 'nanoGPT-Scaling-Laws'...
remote: Enumerating objects: 122, done.
remote: Counting objects: 100% (122/122), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 122 (delta 38), reused 98 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (122/122), 1.28 MiB | 3.54 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/content/nanoGPT-Scaling-Laws
Already on 'main'
Your branch is up to date with 'origin/main'.
From https://github.com/Houcineaberhache/nanoGPT-Scaling-Laws
 * branch            main       -> FETCH_HEAD
Already up to date.


In [3]:
!pip install -q -r requirements.txt

## 2. Pre-flight checks

In [4]:
import os

required_files = [
    "data/processed/train_25.bin",
    "data/processed/train_50.bin",
    "data/processed/train_100.bin",
    "data/processed/val.bin",
    "data/processed/meta.pkl",
    "configs/small.yaml",
    "configs/medium.yaml",
    "configs/experiments.yaml",
    "src/training/train.py",
    "src/training/checkpoint.py",
    "src/training/run_experiments.py",
]

missing = [f for f in required_files if not os.path.exists(f)]
for f in required_files:
    print("OK     " if f not in missing else "MISSING", f)

assert not missing, f"Missing project files: {missing}"

OK      data/processed/train_25.bin
OK      data/processed/train_50.bin
OK      data/processed/train_100.bin
OK      data/processed/val.bin
OK      data/processed/meta.pkl
OK      configs/small.yaml
OK      configs/medium.yaml
OK      configs/experiments.yaml
OK      src/training/train.py
OK      src/training/checkpoint.py
OK      src/training/run_experiments.py


In [5]:
# Team test-suite (17 tests) - should be all green before spending GPU time
!python -m pytest -q

.................                                                        [100%]
17 passed in 9.18s


## 3. Google Drive & experiment output paths

Checkpoints and logs are written to **Google Drive** so nothing is lost if the Colab session disconnects.

> ⚠️ The cell below rewrites `configs/experiments.yaml` **inside the Colab runtime only** (Drive paths).
> Do **not** commit that modification — teammates run the project locally with different paths.

In [6]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [7]:
import os, yaml

DRIVE_RESULTS = "/content/drive/MyDrive/nanoGPT-Scaling-Laws/results"
os.makedirs(DRIVE_RESULTS, exist_ok=True)

config_path = "configs/experiments.yaml"
with open(config_path) as f:
    experiments_config = yaml.safe_load(f)

for name, exp in experiments_config["experiments"].items():
    exp["output"] = f"{DRIVE_RESULTS}/{name}"
    os.makedirs(exp["output"], exist_ok=True)

with open(config_path, "w") as f:
    yaml.safe_dump(experiments_config, f, sort_keys=False)

print("Training budget (shared by all runs):")
for k, v in experiments_config["training"].items():
    print(f"  {k}: {v}")

print("\nOutput directories:")
for name, exp in experiments_config["experiments"].items():
    print(f"  {name} -> {exp['output']}")


def list_outputs(exp):
    """Show the files produced by one experiment (best.pt, latest.pt, metrics.csv)."""
    folder = f"{DRIVE_RESULTS}/{exp}"
    for filename in sorted(os.listdir(folder)):
        size = os.path.getsize(os.path.join(folder, filename)) / 1024**2
        print(f"{filename:15} {size:7.2f} MB")

Training budget (shared by all runs):
  max_iters: 5000
  batch_size: 32
  learning_rate: 0.0003
  eval_interval: 250
  eval_iters: 20
  log_interval: 50
  save_interval: 500
  seed: 42
  device: auto

Output directories:
  E1 -> /content/drive/MyDrive/nanoGPT-Scaling-Laws/results/E1
  E2 -> /content/drive/MyDrive/nanoGPT-Scaling-Laws/results/E2
  E3 -> /content/drive/MyDrive/nanoGPT-Scaling-Laws/results/E3
  E4 -> /content/drive/MyDrive/nanoGPT-Scaling-Laws/results/E4
  E5 -> /content/drive/MyDrive/nanoGPT-Scaling-Laws/results/E5
  E6 -> /content/drive/MyDrive/nanoGPT-Scaling-Laws/results/E6


## 4. Training runs

One cell per run so a Colab disconnection only costs the current experiment.
Approximate time on a T4: **~4 min** for each Small run, **~14 min** for each Medium run.

### E1 — Small • 25 %

In [ ]:
!python -m src.training.run_experiments --only E1 --device cuda
list_outputs("E1")

### E2 — Small • 50 %

In [ ]:
!python -m src.training.run_experiments --only E2 --device cuda
list_outputs("E2")

### E3 — Small • 100 %

In [ ]:
!python -m src.training.run_experiments --only E3 --device cuda
list_outputs("E3")

### E4 — Medium • 25 %

In [ ]:
!python -m src.training.run_experiments --only E4 --device cuda
list_outputs("E4")

### E5 — Medium • 50 %

In [ ]:
!python -m src.training.run_experiments --only E5 --device cuda
list_outputs("E5")

### E6 — Medium • 100 %

In [ ]:
!python -m src.training.run_experiments --only E6 --device cuda
list_outputs("E6")

## 5. Training logs summary

Reads the `metrics.csv` written by each run (losses + elapsed time).
This is the Training Engineer's deliverable: **6 models + checkpoints + logs**.

In [8]:
import pandas as pd

experiments = {
    "E1": ("Small", "25%"),  "E2": ("Small", "50%"),  "E3": ("Small", "100%"),
    "E4": ("Medium", "25%"), "E5": ("Medium", "50%"), "E6": ("Medium", "100%"),
}

rows = []
for exp, (model, data) in experiments.items():
    df = pd.read_csv(f"{DRIVE_RESULTS}/{exp}/metrics.csv")
    best = df.loc[df["val_loss"].idxmin()]
    last = df.iloc[-1]
    rows.append({
        "Run": exp, "Model": model, "Data": data,
        "Iterations": int(last["iteration"]),
        "Best iter": int(best["iteration"]),
        "Final train loss": round(last["train_loss"], 4),
        "Best val loss": round(best["val_loss"], 4),
        "Final val loss": round(last["val_loss"], 4),
        "Train time (s)": round(last["elapsed_seconds"], 1),
    })

training_summary = pd.DataFrame(rows)
display(training_summary)

training_summary.to_csv(f"{DRIVE_RESULTS}/training_summary.csv", index=False)
print("Saved to:", f"{DRIVE_RESULTS}/training_summary.csv")

,Run,Model,Data,Iterations,Best iter,Final train loss,Best val loss,Final val loss,Train time (s)
0,E1,Small,25%,5000,4750,1.3807,2.0041,2.0085,226.0
1,E2,Small,50%,5000,4750,1.4363,1.9314,1.9376,228.6
2,E3,Small,100%,5000,5000,1.5197,1.6743,1.6743,228.7
3,E4,Medium,25%,5000,2250,0.4576,1.9634,2.5696,849.9
4,E5,Medium,50%,5000,2250,0.9915,1.8146,1.8879,853.3
5,E6,Medium,100%,5000,5000,1.1966,1.4648,1.4648,853.2


Saved to: /content/drive/MyDrive/nanoGPT-Scaling-Laws/results/training_summary.csv


## ✅ Hand-off

- 6 checkpoints (`best.pt`, `latest.pt`) and `metrics.csv` per run are on Google Drive under `nanoGPT-Scaling-Laws/results/E1…E6`.
- ➡️ **Next step (Evaluation Engineer):** test loss / perplexity, learning curves, generated samples, baseline comparison and scaling-law analysis — see `notebooks/evaluation_colab.ipynb`.